# Lesson 01 - Bevezetés az AI ügynökökbe

Üdvözlünk az **AI ügynökök kezdőknek** tanfolyam első leckéjében!

Egy **AI ügynök** egy olyan program, amely egy nagy nyelvi modellt (LLM) használ gondolkodási motorjaként, és *műveleteket* hajthat végre a való világban — API-k hívása, adatbázisok lekérdezése vagy kód futtatása — egy felhasználó nevében cél elérése érdekében.

Ebben a jegyzetfüzetben elkészíted az első ügynöködet: egy **Utazási Ügynököt**, amely vakációs úti célokat ajánl. Útközben megtanulod, hogyan:

1. Csatlakozz az Azure AI Foundry Agent Service-hez a **Microsoft Agent Framework** segítségével.
2. Adj az ügynöknek egy **eszközt** — egy egyszerű Python függvényt, amit hívhat.
3. Futtasd az ügynököt és nézd meg a válaszát.
4. Tokenenként streameld az ügynök válaszát.


## Beállítás

A jegyzetfüzet futtatása előtt győződjön meg róla, hogy rendelkezik:

1. **Egy Azure AI Foundry projekttel**, amelyben egy telepített csevegőmodell található (pl. `gpt-4o-mini`).
2. **Bejelentkezett az Azure CLI-be** — futtassa a `az login` parancsot a terminálban.
3. **Beállította a szükséges környezeti változókat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — az Azure AI Foundry projekt végpontja.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modell neve.

Az alábbi cella telepíti a szükséges Python csomagokat.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Az első ügynök létrehozása

Egy ügynöknek két dologra van szüksége:

- **Utasítások**, amelyek megmondják neki, *ki ő* és *hogyan viselkedjen* (egy rendszer parancsa).
- **Eszközök** — Python függvények, amelyeket `@tool` dekorátorral jelöltek, és amelyeket az ügynök hívhat meg információk lekérésére vagy műveletek végrehajtására.

Az alábbiakban definiálunk egy egyszerű eszközt, amely népszerű nyaralási célpontok listáját adja vissza. Az ügynök ezt az eszközt használja majd, amikor egy felhasználó utazási ajánlásokat kér.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Folyamatos válaszok

Az interaktívabb élmény érdekében **folyamatosan** is megjelenítheted az ügynök válaszát. Ahelyett, hogy megvárnád a teljes választ, az ügynök szövegrészeket ad vissza, amint azok elkészülnek. Ez különösen hasznos csevegőfelületeken, ahol a kimenetet valós időben szeretnéd megjeleníteni.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

- **Hozz létre egy szolgáltatót**, amely az Azure AI Foundry Agent Service-hez csatlakozik az `AzureAIProjectAgentProvider` segítségével.
- **Határozz meg egy eszközt** az `@tool` dekorátor használatával, hogy az ügynök hívhassa a Python függvényeidet.
- **Futtasd az ügynököt** egy felhasználói üzenettel, és írd ki a válaszát.
- **Adatfolyamként jelenítsd meg a válaszokat** valós idejű kimenethez.

A következő leckében mélyebben megvizsgáljuk az ügynöki keretrendszereket, és megtanuljuk, hogyan adhatsz az ügynököknek erőteljesebb eszközöket és többlépcsős érvelési képességeket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ezt a dokumentumot az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár az pontos fordításra törekszünk, kérjük vegye figyelembe, hogy az automatikus fordítások tartalmazhatnak hibákat vagy pontatlanságokat. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális, emberi fordítást javaslunk. Nem vállalunk felelősséget semmilyen félreértésért vagy helytelen értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
